# Flujo de modelado

Este cuaderno recorre la carga mensual, el Ridge directo y la lectura del backtest.
El estudio completo, con los modelos alternativos, los gráficos y el informe, se genera con `python -m src.analisis`.

La salida es una proyección estadística. No es asesoría financiera.


In [ ]:
from src.backtest import calcular_metricas, correr_backtest
from src.config import FEATURES_AGREGADOS, FEATURES_TASA_POLITICA, HORIZONTES_REPORTE
from src.data_loader import evaluar_calidad
from src.features import cargar_panel
from src.models import PaseoAleatorio, RidgeDirecto, ajustar_ridge

panel = cargar_panel()
print(len(panel), panel.index.min().date(), panel.index.max().date())
panel.loc[:, ["y", "pol", "g_m1", "g_m2", "g_m3", "inflacion", "brecha_inflacion"]].tail(3)


In [ ]:
calidad = evaluar_calidad()
{
    "observaciones": calidad["muestra"]["observaciones"],
    "tasa_cierre": calidad["muestra"]["tasa_cierre"],
    "politica_cierre": calidad["muestra"]["politica_cierre"],
    "octubre": calidad["parcial_octubre_2026"],
    "decision": calidad["decision_frecuencia"],
}


In [ ]:
origen = len(panel) - 1
for horizonte in HORIZONTES_REPORTE:
    ajuste = ajustar_ridge(panel, origen, horizonte)
    print(horizonte, round(ajuste.nivel, 2), "pares", ajuste.n)


In [ ]:
modelos = [
    RidgeDirecto("ridge_agregados", FEATURES_AGREGADOS),
    RidgeDirecto("ridge_tasa_politica", FEATURES_TASA_POLITICA),
    PaseoAleatorio(),
]
predicciones = correr_backtest(panel, modelos, horizontes=HORIZONTES_REPORTE)
metricas = calcular_metricas(predicciones, panel)
metricas.pivot(index="modelo", columns="horizonte", values="rmse").round(3)
